In [1]:
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

C:\Users\siddh\TruthLens\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2678.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded


In [3]:
topic_corpus = [

{
    "topic":"South Korea Fusion Reactor",

    "year":2026,

    "entities":[
        "South Korea",
        "KSTAR",
        "Fusion Reactor"
    ],

    "keywords":[
        "fusion",
        "plasma",
        "energy"
    ],

    "evidence":[
        "South Korea achieved a fusion milestone.",
        "KSTAR maintained plasma for 48 seconds.",
        "Researchers reached 100 million degrees."
    ]
},

{
    "topic":"India AI Mission",

    "year":2026,

    "entities":[
        "India",
        "AI",
        "IndiaAI"
    ],

    "keywords":[
        "artificial intelligence",
        "mission",
        "government"
    ],

    "evidence":[
        "India launched IndiaAI Mission.",
        "Government invested in AI ecosystem.",
        "AI summit held in New Delhi."
    ]
}

]

In [4]:
topic_texts = []

for item in topic_corpus:

    text = (
        item["topic"] +
        " " +
        " ".join(item["entities"]) +
        " " +
        " ".join(item["keywords"])
    )

    topic_texts.append(text)

topic_embeddings = model.encode(
    topic_texts
)

In [5]:
dimension = topic_embeddings.shape[1]

topic_index = faiss.IndexFlatL2(
    dimension
)

topic_index.add(
    np.array(topic_embeddings)
)

print(
    "Topics:",
    topic_index.ntotal
)

Topics: 2


In [6]:
def retrieve_topic(claim):

    claim_embedding = model.encode(
        [claim]
    )

    distances, indices = topic_index.search(
        np.array(claim_embedding),
        1
    )

    idx = indices[0][0]

    return topic_corpus[idx]

In [7]:
claim = """
A fusion reactor in South Korea generated sustained energy for 48 hours in 2026
"""

topic = retrieve_topic(
    claim
)

print(topic["topic"])

South Korea Fusion Reactor


In [8]:
def retrieve_topic_evidence(claim):

    topic = retrieve_topic(
        claim
    )

    return topic["evidence"]

In [9]:
claim = """
A fusion reactor in South Korea generated sustained energy for 48 hours in 2026
"""

evidence = retrieve_topic_evidence(
    claim
)

for e in evidence:

    print(e)

South Korea achieved a fusion milestone.
KSTAR maintained plasma for 48 seconds.
Researchers reached 100 million degrees.


In [10]:
claims = [

"A fusion reactor in South Korea generated sustained energy for 48 hours in 2026",

"India launched a nationwide AI surveillance system in 2026",

"The government invested heavily in artificial intelligence through IndiaAI Mission",

"Researchers maintained plasma at 100 million degrees in South Korea",

"A tokamak reactor achieved a major fusion milestone"

]

In [11]:
topic = retrieve_topic(claim)

print("Claim:", claim)
print("Retrieved Topic:", topic["topic"])
print()

Claim: 
A fusion reactor in South Korea generated sustained energy for 48 hours in 2026

Retrieved Topic: South Korea Fusion Reactor

